In [1]:
from physics.simulation import mcfm, msq
from physics.hzz import zz4l
from physics.hstar import eft

import numpy as np
import json
import hist
import matplotlib, matplotlib.pyplot as plt


In [2]:
matplotlib.use("pgf")
matplotlib.rcParams.update({
    "pgf.texsystem": "lualatex",
    "text.usetex": True,
    "pgf.rcfonts": False,
    "pgf.preamble": "", 
})
lw  = 1.0

In [3]:
with open('data/zz4l/xsec.json') as f:
    xsec = json.load(f)

xsec = {
    msq.Component.SBI : np.prod(xsec['gg_sbi']),
    msq.Component.SIG : np.prod(xsec['gg_sig']),
    msq.Component.INT : np.prod(xsec['gg_int']),
    msq.Component.BKG : np.prod(xsec['gg_bkg'])
}

filenames = {
    msq.Component.SBI : 'data/zz4l/ggZZ_sbi/events_*.csv',
    msq.Component.SIG : 'data/zz4l/ggZZ_sig/events_*.csv',
    msq.Component.INT : 'data/zz4l/ggZZ_int/events_*.csv',
    msq.Component.BKG : 'data/zz4l/ggZZ_bkg/events_*.csv'
}

In [4]:
print(mcfm.csv_component_bsm[msq.Component.SBI][(0,-1,0)])

msq_sbi_bsm_20


In [5]:
events_sig = mcfm.from_csv(cross_section=xsec[msq.Component.SIG], file_path=filenames[msq.Component.SIG], n_rows=100_000)
events_sig = zz4l.analyze(events_sig)

events_sbi = mcfm.from_csv(cross_section=xsec[msq.Component.SBI], file_path=filenames[msq.Component.SBI], n_rows=100_000)
events_sbi = zz4l.analyze(events_sbi)

In [6]:
from physics.variables import cH_vals

eft_vals = [+1, -1]

eft_colors = ['red', 'blue']
eft_lines = ['-', '-']

mod_sig = eft.Modifier(baseline=msq.Component.SIG, events=events_sig)
wt_sig_eft, prob_sig_eft = mod_sig.modify(c6 = [0], ct = eft_vals, cg = [0])

mod_sbi = eft.Modifier(baseline=msq.Component.SBI, events=events_sbi)
wt_sbi_eft, prob_sbi_eft = mod_sbi.modify(c6 = [0], ct = eft_vals, cg = [0])

In [22]:
print(mod_sbi.coefficients[0,0,0,0])
print(mod_sbi.coefficients[0,0,1,0])
print(mod_sbi.coefficients[0,0,0,1])
print(mod_sbi.coefficients[0,0,1,1])
print(mod_sbi.coefficients[0,0,2,1])

1.0
0.0780911231825434
0.049415972211286546
0.11120594613124737
-4.326195857107251e-08


In [8]:
color_sm = 'black'
line_sm = '--'

In [64]:
xobs_sbi = events_sbi.kinematics['4l_mass'].to_numpy()
xobs_sig = events_sig.kinematics['4l_mass'].to_numpy()

nxbins = 41
xmin, xmax = 180, 1000.0
xbins = np.linspace(xmin, xmax, nxbins + 1)
xcenters = 0.5 * (xbins[:-1] + xbins[1:])
xwidths = np.diff(xbins)

In [65]:
fb_to_ab = 1000.0
h_sig_sm = hist.Hist(hist.axis.Variable(xbins), storage=hist.storage.Weight())
h_sig_sm.fill(xobs_sig, weight=events_sig.weights * fb_to_ab)

h_sig_eft = []
for i_eft, eft_val in enumerate(eft_vals):
    h = hist.Hist(hist.axis.Variable(xbins), storage=hist.storage.Weight())
    h.fill(xobs_sig, weight=wt_sig_eft[:,0,i_eft,0] * fb_to_ab)
    h_sig_eft.append(h)

h_sbi_sm = hist.Hist(hist.axis.Variable(xbins), storage=hist.storage.Weight())
h_sbi_sm.fill(xobs_sbi, weight=events_sbi.weights)

h_sbi_eft = []
for i_eft, eft_val in enumerate(eft_vals):
    h = hist.Hist(hist.axis.Variable(xbins), storage=hist.storage.Weight())
    h.fill(xobs_sbi, weight=wt_sbi_eft[:,0,i_eft,0])
    h_sbi_eft.append(h)


In [66]:
fig, (ax1, ax2) = plt.subplots(2,1, gridspec_kw={'height_ratios': [3,1]}, figsize=(5,5), sharex=True)

l_eft = []
for i_eft, eft_val in enumerate(eft_vals):
    l_eft.append(ax1.stairs(h_sbi_eft[i_eft].values(), xbins, color=eft_colors[i_eft], linestyle=eft_lines[i_eft], linewidth=lw))
l_sm = ax1.stairs(h_sbi_sm.values(), xbins, color=color_sm, linestyle=line_sm, linewidth=lw)

l_sm.set_label('$\mathrm{SM}$')
for i_eft, eft_val in enumerate(eft_vals):
    sgn = '-' if eft_val < 0 else '+'
    l_eft[i_eft].set_label(f'$C_{{H}} = {sgn}{abs(cH_vals[i_eft]):d}$')
ax1.legend(frameon=False, fontsize=12)

for i_eft, eft_val in enumerate(eft_vals):
    ax2.stairs(h_sbi_eft[i_eft].values() / h_sbi_sm.values(), xbins, color=eft_colors[i_eft], linestyle=eft_lines[i_eft], linewidth=lw)
ax2.stairs(h_sbi_sm.values() / h_sbi_sm.values(), xbins, color=color_sm, linestyle=line_sm, linewidth=lw)

ax1.set_yscale('log')
ax1.set_ylabel('$\\mathrm{d}\sigma / \\mathrm{d} m_{ZZ}\\ \\mathrm{[fb / GeV]}$', loc='top', fontsize=15)
ax1.set_ylim(1e-4, 10.0)
ax1.yaxis.set_ticks([1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0])
ax1.yaxis.set_ticklabels(['$10^{-4}$', '$10^{-3}$', '$10^{-2}$', '$0.1$', '$1.0$', '$10$'])

ax1.text(0.04 ,0.12, '$gg (\\rightarrow h^{\\ast}) \\rightarrow ZZ$', transform=ax1.transAxes, ha='left', va='bottom', fontsize=12)
ax1.text(0.04 ,0.04, '$\\sqrt{s} = 14\\,\\mathrm{TeV},\\  3\\, \\mathrm{ab}^{-1}$', transform=ax1.transAxes, ha='left', va='bottom', fontsize=12)

ax2.set_ylabel('$\\mathrm{BSM} / \\mathrm{SM}$', fontsize=15)
ax2.set_ylim(0.95,1.25)

ax2.set_xscale('linear')
ax2.set_xlim(xmin, xmax)
ax2.set_xlabel('$m_{ZZ}\\ \\mathrm{[GeV]}$', loc='right', fontsize=15)
ax2.tick_params(top=True, labeltop=False)
ax2.xaxis.set_ticks([180, 250, 500, 750, 1000])

fig.tight_layout()
fig.subplots_adjust(hspace=0)

fig.canvas.draw()  # update positions
ax1_pos, ax2_pos = ax1.get_position(), ax2.get_position()
ax2.set_position([ax1_pos.x0, ax2_pos.y0, ax1_pos.width, ax2_pos.height]) # align 2nd x-axis with 1st

plt.savefig('sbi_4l_mzz.pdf')
plt.close()

In [67]:
fig, (ax1, ax2) = plt.subplots(2,1, gridspec_kw={'height_ratios': [3,1]}, figsize=(5,5), sharex=True)

l_eft = []
for i_eft, eft_val in enumerate(eft_vals):
    l_eft.append(ax1.stairs(h_sig_eft[i_eft].values(), xbins, color=eft_colors[i_eft], linestyle=eft_lines[i_eft], linewidth=lw))
l_sm = ax1.stairs(h_sig_sm.values(), xbins, color=color_sm, linestyle=line_sm, linewidth=lw)

l_sm.set_label('$\mathrm{SM}$')
for i_eft, eft_val in enumerate(eft_vals):
    sgn = '-' if eft_val < 0 else '+'
    l_eft[i_eft].set_label(f'$C_{{H}} = {sgn}{abs(eft_val):d}$')
ax1.legend(frameon=False, fontsize=12)

for i_eft, eft_val in enumerate(eft_vals):
    ax2.stairs(h_sig_eft[i_eft].values() / h_sig_sm.values(), xbins, color=eft_colors[i_eft], linestyle=eft_lines[i_eft], linewidth=lw)
ax2.stairs(h_sig_sm.values() / h_sig_sm.values(), xbins, color=color_sm, linestyle=line_sm, linewidth=lw)

ax1.set_yscale('linear')
ax1.set_ylabel('$\\mathrm{d}\sigma / \\mathrm{d} m_{ZZ}\\ \\mathrm{[ab / GeV]}$', loc='top', fontsize=15)
ax1.set_ylim(0.0, 25.0)
ax1.tick_params(labelsize=12)

ax1.text(0.04 ,0.96, '$gg \\rightarrow h^{\\ast} \\rightarrow ZZ$', transform=ax1.transAxes, ha='left', va='top', fontsize=12)
ax1.text(0.04 ,0.88, '$\\sqrt{s} = 14\\,\\mathrm{TeV},\\  3\\, \\mathrm{ab}^{-1}$', transform=ax1.transAxes, ha='left', va='top', fontsize=12)

ax2.set_ylabel('$\\mathrm{BSM} / \\mathrm{SM}$', fontsize=15)
ax2.set_ylim(0.0,2.0)
ax2.yaxis.set_ticks([0.0, 0.5, 1.0, 1.5])

ax2.set_xscale('linear')
ax2.set_xlim(xmin, xmax)
ax2.set_xlabel('$m_{ZZ}\\ \\mathrm{[GeV]}$', loc='right', fontsize=15)
ax2.tick_params(top=True, labeltop=False, labelsize=12)

fig.tight_layout()
fig.subplots_adjust(hspace=0)

fig.canvas.draw()  # update positions
ax1_pos, ax2_pos = ax1.get_position(), ax2.get_position()
ax2.set_position([ax1_pos.x0, ax2_pos.y0, ax1_pos.width, ax2_pos.height]) # align 2nd x-axis with 1st

plt.savefig('sig_4l_mzz.pdf')
plt.close()